In [ ]:
%pip install -q "datasets>=3.0,<5" "huggingface_hub>=0.25" ijson tqdm "bert-score>=0.3.13,<0.4"

In [ ]:
import os, re, json, math, time, hashlib, random, gc, zipfile, unicodedata, heapq
from pathlib import Path
from getpass import getpass
from dataclasses import dataclass
from typing import Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import ijson
from huggingface_hub import HfFileSystem

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATASET_CONFIG = "phase_1"
SOURCES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}
TRAIN_TOKEN_BUDGET = 3_000_000
VALID_TOKEN_BUDGET = 100_000
VALID_FRACTION = 0.02
MAX_DOCS_PER_SOURCE = 50_000
MAX_CHARS_PER_DOC = 50_000

RECENT_TOKENS = 256
OLD_HISTORY_TOKENS = 2_048
COMPRESS_GROUP_SIZE = 8
COMPRESSED_SLOTS = OLD_HISTORY_TOKENS // COMPRESS_GROUP_SIZE
SEGMENT_LENGTH = 256
EPOCHS = 4
GRAD_ACCUM = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
WARMUP_FRACTION = 0.05

MODEL_D = 768
MODEL_LAYERS = 12
QUERY_HEADS = 12
KV_HEADS = 6
FF_DIM = 3072
DROPOUT = 0.10
ROPE_BASE = 10_000.0
RMS_EPS = 1e-6

OUTPUT_WORDS = 128
MAX_GENERATION_TOKENS = 1024
GEN_TEMPERATURE, GEN_TOP_K, GEN_TOP_P = 0.8, 50, 0.90
RUN_OFFICIAL_GENERATION = False

OUTPUT_DIR = Path("/kaggle/working/phase1_multires_memory_outputs")
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_OVERRIDE = os.environ.get("LEGAL_BPE_TOKENIZER_PATH")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face read token for the gated official dataset: ")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required to read the gated official dataset.")

In [ ]:
class ByteBPE:
    def __init__(self, obj, cache_limit=50_000):
        self.itos = list(obj["itos"])
        self.vocab_size = int(obj["vocab_size"])
        self.merges = [tuple(map(int, p)) for p in obj["merges"]]
        if self.itos[:4] != ["<pad>", "<unk>", "<s>", "</s>"] or len(self.itos) != self.vocab_size:
            raise ValueError("Unsupported ByteBPE vocabulary layout.")
        if 261 + len(self.merges) != self.vocab_size:
            raise ValueError("Tokenizer merge count and vocabulary size disagree.")
        self.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        self.payload[260] = b""
        for rank, (a, b) in enumerate(self.merges):
            self.payload[261 + rank] = self.payload.get(a, b"") + self.payload.get(b, b"")
        self.ranks = {p: r for r, p in enumerate(self.merges)}
        self.cache_limit, self.cache = cache_limit, {}
        self.word_end = [False] * self.vocab_size
        self.word_end[260] = True
        for r, (a, b) in enumerate(self.merges):
            self.word_end[261 + r] = self.word_end[a] or self.word_end[b]

    def encode_word(self, word):
        if word in self.cache:
            return list(self.cache[word])
        symbols = [b + 4 for b in word.encode("utf-8")] + [260]
        n = len(symbols)
        left, right = [i-1 for i in range(n)], [i+1 for i in range(n)]
        right[-1] = -1
        alive, version = [True]*n, [0]*n
        heap = []
        def push(i):
            if i < 0 or not alive[i]: return
            j = right[i]
            if j >= 0 and alive[j]:
                rank = self.ranks.get((symbols[i], symbols[j]))
                if rank is not None:
                    heapq.heappush(heap, (rank, i, j, version[i], version[j]))
        for i in range(n-1): push(i)
        while heap:
            rank, i, j, vi, vj = heapq.heappop(heap)
            if not alive[i] or not alive[j] or right[i] != j or version[i] != vi or version[j] != vj:
                continue
            if self.ranks.get((symbols[i], symbols[j])) != rank: continue
            symbols[i] = 261 + rank
            version[i] += 1
            after = right[j]
            right[i] = after
            if after >= 0:
                left[after] = i
            alive[j] = False
            version[j] += 1
            push(left[i]); push(i)
        head = next(i for i in range(n) if alive[i] and left[i] < 0)
        ids, i = [], head
        while i >= 0:
            ids.append(symbols[i]); i = right[i]
        if len(self.cache) < self.cache_limit: self.cache[word] = tuple(ids)
        return ids

    def encode(self, text):
        return [t for word in str(text).split() for t in self.encode_word(word)]

    def decode(self, ids):
        pieces, buf = [], bytearray()
        for token in ids:
            token = int(token)
            if token < 4 or token >= self.vocab_size: continue
            buf.extend(self.payload.get(token, b""))
            if self.word_end[token]:
                pieces.append(buf.decode("utf-8", errors="replace")); buf.clear()
        if buf: pieces.append(buf.decode("utf-8", errors="replace"))
        return " ".join(p for p in pieces if p).strip()

def load_tokenizer():
    paths = []
    if TOKENIZER_OVERRIDE: paths.append(Path(TOKENIZER_OVERRIDE).expanduser())
    paths.extend([Path("legal_custom_bytebpe_32768.json"), Path("legal_bpe_tokenizer.json")])
    root = Path("/kaggle/input")
    if root.exists():
        paths.extend(root.rglob("legal_custom_bytebpe_32768.json"))
        paths.extend(root.rglob("legal_bpe_tokenizer*.json"))
    valid = []
    for path in paths:
        if path.is_file():
            try:
                obj = json.loads(path.read_text(encoding="utf-8"))
                valid.append((path, ByteBPE(obj)))
            except Exception as e:
                print("Ignoring incompatible tokenizer:", path, e)
    if not valid:
        raise FileNotFoundError("Attach your saved ByteBPE JSON to Kaggle Inputs or set LEGAL_BPE_TOKENIZER_PATH.")
    valid.sort(key=lambda x: x[1].vocab_size != 32768)
    return valid[0]

TOKENIZER_PATH, tokenizer = load_tokenizer()
TOKENIZER_SHA256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
print("Tokenizer:", TOKENIZER_PATH, "vocab:", VOCAB_SIZE, "sha256:", TOKENIZER_SHA256)
sample = "The Supreme Court considered the appeal and dismissed the petition."
print("Round-trip:", tokenizer.decode(tokenizer.encode(sample)))

In [ ]:
def extract_text(record):
    keys = ("text","content","document","judgment","judgement","body","full_text","prompt","prompt_text","input","case_text")
    if isinstance(record, str): return record
    if isinstance(record, dict):
        for key in keys:
            value = record.get(key)
            if isinstance(value, str) and value.strip(): return value
        for key, value in record.items():
            if str(key).lower() in {"id","document_id","doc_id","example_id"}: continue
            if isinstance(value, str) and len(value.strip()) > 20: return value
            if isinstance(value, (dict,list,tuple)):
                got = extract_text(value)
                if got: return got
    if isinstance(record, (list,tuple)):
        for value in record:
            got = extract_text(value)
            if got: return got
    return ""

def clean_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def flatten_docs(docs):
    arrays = [np.asarray(d, dtype=np.int32) for d in docs if len(d) >= 2]
    flat = np.concatenate(arrays) if arrays else np.empty(0, dtype=np.int32)
    offsets = np.zeros(len(arrays)+1, dtype=np.int64)
    for i,a in enumerate(arrays): offsets[i+1] = offsets[i] + len(a)
    return flat, offsets

def save_docs(path, docs):
    flat, offsets = flatten_docs(docs)
    np.savez_compressed(path, tokens=flat, offsets=offsets)

def read_docs(path):
    with np.load(path, allow_pickle=False) as z:
        flat, offsets = z["tokens"], z["offsets"]
    return [flat[offsets[i]:offsets[i+1]].copy() for i in range(len(offsets)-1)]

cache_args = dict(dataset=DATASET_ID, tokenizer=TOKENIZER_SHA256, train=TRAIN_TOKEN_BUDGET,
                  valid=VALID_TOKEN_BUDGET, fraction=VALID_FRACTION, seed=SEED, max_chars=MAX_CHARS_PER_DOC)
CACHE_SIG = hashlib.sha256(json.dumps(cache_args, sort_keys=True).encode()).hexdigest()
TRAIN_CACHE, VALID_CACHE = CACHE_DIR/"train.npz", CACHE_DIR/"valid.npz"
META_CACHE = CACHE_DIR/"manifest.json"

def build_splits():
    fs = HfFileSystem(token=HF_TOKEN, block_size=1024*1024)
    train, valid, counts = [], [], {}
    train_cap = TRAIN_TOKEN_BUDGET // len(SOURCES)
    valid_cap = VALID_TOKEN_BUDGET // len(SOURCES)
    for name, relpath in SOURCES.items():
        ntrain = nvalid = scanned = 0
        train_local, valid_local = [], []
        remote = f"datasets/{DATASET_ID}/{relpath}"
        print("Streaming", name, relpath)
        try:
            with fs.open(remote, "rb") as fh:
                for row_idx, row in enumerate(ijson.items(fh, "item")):
                    if scanned >= MAX_DOCS_PER_SOURCE: break
                    scanned += 1
                    text = clean_text(extract_text(row))[:MAX_CHARS_PER_DOC]
                    if len(text) < 20: continue
                    heldout = ((row_idx + SEED) % 10_000) < int(VALID_FRACTION*10_000)
                    if heldout and nvalid >= valid_cap: continue
                    if not heldout and ntrain >= train_cap: continue
                    ids = tokenizer.encode(text)
                    if len(ids) < 2: continue
                    if heldout:
                        remaining = valid_cap - nvalid
                        if remaining <= 1: continue
                        ids = ids[:remaining-1] + [EOS_ID]
                        valid_local.append(ids); nvalid += len(ids)
                    else:
                        remaining = train_cap - ntrain
                        if remaining <= 1: continue
                        ids = ids[:remaining-1] + [EOS_ID]
                        train_local.append(ids); ntrain += len(ids)
                    if ntrain >= train_cap and nvalid >= valid_cap: break
        except Exception as exc:
            raise RuntimeError(f"Could not read {relpath}; check HF_TOKEN and gated dataset access.") from exc
        train.extend(train_local); valid.extend(valid_local)
        counts[name] = {"scanned": scanned, "train_docs": len(train_local), "train_tokens": ntrain,
                        "valid_docs": len(valid_local), "valid_tokens": nvalid}
        print(counts[name])
    if not train or not valid: raise RuntimeError("Train or validation split is empty.")
    save_docs(TRAIN_CACHE, train); save_docs(VALID_CACHE, valid)
    META_CACHE.write_text(json.dumps({"signature": CACHE_SIG, "counts": counts}, indent=2), encoding="utf-8")
    return train, valid, counts

cache_valid = False
if TRAIN_CACHE.exists() and VALID_CACHE.exists() and META_CACHE.exists():
    try: cache_valid = json.loads(META_CACHE.read_text(encoding="utf-8")).get("signature") == CACHE_SIG
    except Exception: cache_valid = False
if cache_valid:
    train_documents, valid_documents = read_docs(TRAIN_CACHE), read_docs(VALID_CACHE)
    source_counts = json.loads(META_CACHE.read_text(encoding="utf-8")).get("counts", {})
    print("Loaded cached tokenized corpus.")
else:
    train_documents, valid_documents, source_counts = build_splits()
print("Train documents/tokens:", len(train_documents), sum(map(len,train_documents)))
print("Validation documents/tokens:", len(valid_documents), sum(map(len,valid_documents)))
for idx, doc in enumerate(train_documents[:2]):
    print(f"Sample {idx+1}:", tokenizer.decode(doc[:160]))

In [ ]:
@dataclass
class LayerMemory:
    compressed: Optional[torch.Tensor] = None
    compressed_pos: Optional[torch.Tensor] = None
    recent: Optional[torch.Tensor] = None
    recent_pos: Optional[torch.Tensor] = None
    pending: Optional[torch.Tensor] = None
    pending_pos: Optional[torch.Tensor] = None

@dataclass
class LayerCache:
    compressed_k: Optional[torch.Tensor] = None
    compressed_v: Optional[torch.Tensor] = None
    pending_k: Optional[torch.Tensor] = None
    pending_v: Optional[torch.Tensor] = None
    recent_k: Optional[torch.Tensor] = None
    recent_v: Optional[torch.Tensor] = None

def cat_time(a,b):
    if a is None or a.size(1) == 0: return b
    if b is None or b.size(1) == 0: return a
    return torch.cat((a,b), dim=1)

def update_memory(old, current, positions):
    if old is None: old = LayerMemory()
    joined = cat_time(old.recent, current)
    joined_pos = cat_time(old.recent_pos, positions)
    evict = max(0, joined.size(1)-RECENT_TOKENS)
    if evict:
        dropped, dropped_pos = joined[:,:evict], joined_pos[:,:evict]
        recent, recent_pos = joined[:,-RECENT_TOKENS:], joined_pos[:,-RECENT_TOKENS:]
    else:
        dropped, dropped_pos = joined[:,:0], joined_pos[:,:0]
        recent, recent_pos = joined, joined_pos
    pending = cat_time(old.pending, dropped)
    pending_pos = cat_time(old.pending_pos, dropped_pos)
    groups = pending.size(1)//COMPRESS_GROUP_SIZE
    n = groups*COMPRESS_GROUP_SIZE
    if groups:
        new_comp = pending[:,:n].reshape(pending.size(0),groups,COMPRESS_GROUP_SIZE,pending.size(-1)).mean(2)
        new_pos = pending_pos[:,:n].reshape(pending_pos.size(0),groups,COMPRESS_GROUP_SIZE).mean(2)
        pending, pending_pos = pending[:,n:], pending_pos[:,n:]
    else:
        new_comp, new_pos = pending[:,:0], pending_pos[:,:0]
    comp = cat_time(old.compressed,new_comp)
    comp_pos = cat_time(old.compressed_pos,new_pos)
    if comp.size(1)>COMPRESSED_SLOTS:
        comp, comp_pos = comp[:,-COMPRESSED_SLOTS:], comp_pos[:,-COMPRESSED_SLOTS:]
    result = LayerMemory(comp.detach(), comp_pos.detach(), recent.detach(), recent_pos.detach(),
                         pending.detach(), pending_pos.detach())
    return result, new_comp.detach(), new_pos.detach()

class RMSNorm(nn.Module):
    def __init__(self, width, eps=RMS_EPS):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps
    def forward(self,x):
        norm = torch.rsqrt(x.float().pow(2).mean(-1,keepdim=True)+self.eps)
        return (x.float()*norm).to(x.dtype)*self.weight.to(x.dtype)

class RoPE(nn.Module):
    def __init__(self, dim):
        super().__init__()
        inv = 1.0/(ROPE_BASE**(torch.arange(0,dim,2,dtype=torch.float32)/dim))
        self.register_buffer("inv",inv,persistent=False)
    def rotate(self,x,pos):
        ang = pos.to(x.device,dtype=torch.float32).unsqueeze(-1)*self.inv
        cos = torch.cat((ang.cos(), ang.cos()), dim=-1).unsqueeze(1).to(x.dtype)
        sin = torch.cat((ang.sin(), ang.sin()), dim=-1).unsqueeze(1).to(x.dtype)
        half=x.size(-1)//2
        rot=torch.cat((-x[...,half:],x[...,:half]),-1)
        return x*cos+rot*sin

class GQA(nn.Module):
    def __init__(self):
        super().__init__()
        if MODEL_D%QUERY_HEADS or QUERY_HEADS%KV_HEADS: raise ValueError("Invalid GQA dimensions.")
        self.dh=MODEL_D//QUERY_HEADS
        self.groups=QUERY_HEADS//KV_HEADS
        self.q=nn.Linear(MODEL_D,QUERY_HEADS*self.dh,bias=False)
        self.k=nn.Linear(MODEL_D,KV_HEADS*self.dh,bias=False)
        self.v=nn.Linear(MODEL_D,KV_HEADS*self.dh,bias=False)
        self.o=nn.Linear(MODEL_D,MODEL_D,bias=False)
        self.rope=RoPE(self.dh)
    def project(self,x,pos):
        b,t,_=x.shape
        q=self.q(x).view(b,t,QUERY_HEADS,self.dh).transpose(1,2)
        k=self.k(x).view(b,t,KV_HEADS,self.dh).transpose(1,2)
        v=self.v(x).view(b,t,KV_HEADS,self.dh).transpose(1,2)
        return self.rope.rotate(q,pos),self.rope.rotate(k,pos),v
    def project_memory(self,x,pos):
        if x is None or x.size(1)==0: return None,None
        b,t,_=x.shape
        k=self.k(x).view(b,t,KV_HEADS,self.dh).transpose(1,2)
        v=self.v(x).view(b,t,KV_HEADS,self.dh).transpose(1,2)
        return self.rope.rotate(k,pos),v
    def attend(self,q,k,v,past_len,dropout):
        k=k.repeat_interleave(self.groups,dim=1)
        v=v.repeat_interleave(self.groups,dim=1)
        t=q.size(-2); total=k.size(-2)
        allowed=torch.ones((t,total),dtype=torch.bool,device=q.device)
        allowed[:,past_len:]=torch.tril(torch.ones((t,t),dtype=torch.bool,device=q.device))
        out=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed[None,None],is_causal=False,
                                           dropout_p=dropout if self.training else 0.0)
        return self.o(out.transpose(1,2).contiguous().view(q.size(0),t,MODEL_D))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1=RMSNorm(MODEL_D); self.attn=GQA(); self.n2=RMSNorm(MODEL_D)
        self.ff1=nn.Linear(MODEL_D,FF_DIM); self.ff2=nn.Linear(FF_DIM,MODEL_D)
        self.drop=nn.Dropout(DROPOUT)
    def forward(self,x,pos,memory=None,cache=None):
        n=self.n1(x); q,ck,cv=self.attn.project(n,pos)
        if cache is not None:
            pk=torch.cat((cache.compressed_k,cache.pending_k,cache.recent_k),dim=-2)
            pv=torch.cat((cache.compressed_v,cache.pending_v,cache.recent_v),dim=-2)
        else:
            pk=pv=None
            if memory is not None:
                mh=cat_time(memory.compressed,cat_time(memory.pending,memory.recent))
                mp=cat_time(memory.compressed_pos,cat_time(memory.pending_pos,memory.recent_pos))
                if mh is not None and mh.size(1): pk,pv=self.attn.project_memory(self.n1(mh),mp)
        if pk is None:
            pk=ck[:,:,:0]; pv=cv[:,:,:0]
        past=pk.size(-2)
        y=self.attn.attend(q,torch.cat((pk,ck),-2),torch.cat((pv,cv),-2),past,DROPOUT)
        x=x+self.drop(y)
        x=x+self.drop(self.ff2(F.gelu(self.ff1(self.n2(x)))))
        return x,ck,cv
    def project_summary(self,hidden,pos):
        return self.attn.project_memory(self.n1(hidden),pos)

class MultiResolutionTransformerXL(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.emb=nn.Embedding(vocab,MODEL_D)
        self.layers=nn.ModuleList([Block() for _ in range(MODEL_LAYERS)])
        self.final_norm=RMSNorm(MODEL_D)
        self.lm_head=nn.Linear(MODEL_D,vocab,bias=False)
        self.apply(self.init_weights)
        self.lm_head.weight=self.emb.weight
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m,RMSNorm): nn.init.ones_(m.weight)
    def forward(self,ids,memories=None,caches=None,position_start=0,use_cache=False):
        b,t=ids.shape
        pos=torch.arange(position_start,position_start+t,device=ids.device,dtype=torch.float32)[None,:].expand(b,-1)
        memories=[None]*len(self.layers) if memories is None else memories
        caches=([None]*len(self.layers) if caches is None else caches) if use_cache else None
        x=self.emb(ids); new_mem=[]; new_cache=[] if use_cache else None
        for i,layer in enumerate(self.layers):
            layer_input=x
            x,cur_k,cur_v=layer(x,pos,memories[i],caches[i] if use_cache else None)
            mem,new_summary,new_summary_pos=update_memory(memories[i],layer_input,pos)
            new_mem.append(mem)
            if use_cache:
                old=caches[i]
                if old is None:
                    empty=cur_k[:,:,:0]
                    old_ck=old_cv=old_pk=old_pv=old_rk=old_rv=empty
                else:
                    old_ck,old_cv=old.compressed_k,old.compressed_v
                    old_pk,old_pv=old.pending_k,old.pending_v
                    old_rk,old_rv=old.recent_k,old.recent_v
                if new_summary.size(1):
                    sk,sv=layer.project_summary(new_summary,new_summary_pos)
                    comp_k=torch.cat((old_ck,sk),-2)[:,:,-COMPRESSED_SLOTS:]
                    comp_v=torch.cat((old_cv,sv),-2)[:,:,-COMPRESSED_SLOTS:]
                else: comp_k,comp_v=old_ck,old_cv
                joined_rk=torch.cat((old_rk,cur_k),-2)
                joined_rv=torch.cat((old_rv,cur_v),-2)
                evict=max(0,joined_rk.size(-2)-RECENT_TOKENS)
                evicted_k=joined_rk[:,:,:evict]
                evicted_v=joined_rv[:,:,:evict]
                recent_k=joined_rk[:,:,-RECENT_TOKENS:]
                recent_v=joined_rv[:,:,-RECENT_TOKENS:]
                joined_pk=torch.cat((old_pk,evicted_k),-2)
                joined_pv=torch.cat((old_pv,evicted_v),-2)
                consumed=new_summary.size(1)*COMPRESS_GROUP_SIZE
                pending_k=joined_pk[:,:,consumed:]
                pending_v=joined_pv[:,:,consumed:]
                new_cache.append(LayerCache(comp_k,comp_v,pending_k,pending_v,recent_k,recent_v))
        return self.lm_head(self.final_norm(x)),new_mem,new_cache

MODEL=MultiResolutionTransformerXL(VOCAB_SIZE).to(DEVICE)
PARAMS=sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print(f"Trainable parameters: {PARAMS:,} ({PARAMS/1e6:.2f}M)")
if not 70_000_000<=PARAMS<=250_000_000: raise ValueError("Model must contain 70M-250M trainable parameters.")

In [ ]:
BEST_PATH=OUTPUT_DIR/"best_multires_gqa_rmsnorm.pt"
LATEST_PATH=OUTPUT_DIR/"latest_multires_gqa_rmsnorm.pt"
USE_AMP=DEVICE.type=="cuda"
SCALER=torch.amp.GradScaler("cuda",enabled=USE_AMP) if USE_AMP else None

def amp():
    return torch.autocast("cuda",dtype=torch.float16) if USE_AMP else torch.autocast("cpu",enabled=False)

def micro_steps(docs):
    return sum(math.ceil((len(d)-1)/SEGMENT_LENGTH) for d in docs if len(d)>1)

def evaluate(model,docs,limit=VALID_TOKEN_BUDGET):
    model.eval(); total_loss=0.0; total=0
    with torch.inference_mode():
        for doc in docs:
            if total>=limit: break
            mem=None; upto=min(len(doc),limit-total+1)
            for start in range(0,upto-1,SEGMENT_LENGTH):
                end=min(start+SEGMENT_LENGTH,upto-1)
                if end<=start: continue
                x=torch.tensor(doc[start:end],dtype=torch.long,device=DEVICE)[None,:]
                y=torch.tensor(doc[start+1:end+1],dtype=torch.long,device=DEVICE)[None,:]
                with amp():
                    logits,mem,_=model(x,memories=mem,position_start=start)
                    loss=F.cross_entropy(logits.float().reshape(-1,VOCAB_SIZE),y.reshape(-1),reduction="sum")
                total_loss+=loss.item(); total+=y.numel()
    model.train()
    return total_loss/max(total,1),total

def save_ckpt(path,model,opt,epoch,best,history):
    payload={
        "model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),
        "epoch":epoch,"best_validation_loss":best,"history":history,
        "tokenizer_sha256":TOKENIZER_SHA256,"vocab_size":VOCAB_SIZE,
        "config":{"d_model":MODEL_D,"layers":MODEL_LAYERS,"query_heads":QUERY_HEADS,
                  "kv_heads":KV_HEADS,"ff_dim":FF_DIM,"recent_tokens":RECENT_TOKENS,
                  "old_history_tokens":OLD_HISTORY_TOKENS,"compressed_slots":COMPRESSED_SLOTS,
                  "compress_group_size":COMPRESS_GROUP_SIZE,"segment_length":SEGMENT_LENGTH,
                  "rope_base":ROPE_BASE,"rms_eps":RMS_EPS},
    }
    tmp=Path(str(path)+".tmp"); torch.save(payload,tmp); tmp.replace(path)

def train(model,train_docs,valid_docs):
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,betas=(0.9,0.95),weight_decay=WEIGHT_DECAY)
    total_updates=max(1,math.ceil(micro_steps(train_docs)*EPOCHS/GRAD_ACCUM))
    warm=max(1,int(total_updates*WARMUP_FRACTION))
    def scale(step):
        if step<warm: return max(1e-3,(step+1)/warm)
        frac=min(1.0,(step-warm)/max(1,total_updates-warm))
        return 0.1+0.9*0.5*(1+math.cos(math.pi*frac))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,scale)
    history=[]; best=float("inf"); opt.zero_grad(set_to_none=True); pending=0; updates=0
    for epoch in range(1,EPOCHS+1):
        model.train(); loss_sum=0.0; token_sum=0
        order=np.random.permutation(len(train_docs))
        for doc_i in tqdm(order,desc=f"Epoch {epoch}/{EPOCHS}"):
            doc=train_docs[int(doc_i)]
            if len(doc)<2: continue
            mem=None
            for start in range(0,len(doc)-1,SEGMENT_LENGTH):
                end=min(start+SEGMENT_LENGTH,len(doc)-1)
                if end<=start: continue
                x=torch.tensor(doc[start:end],dtype=torch.long,device=DEVICE)[None,:]
                y=torch.tensor(doc[start+1:end+1],dtype=torch.long,device=DEVICE)[None,:]
                with amp():
                    logits,mem,_=model(x,memories=mem,position_start=start)
                    loss=F.cross_entropy(logits.float().reshape(-1,VOCAB_SIZE),y.reshape(-1))
                if SCALER: SCALER.scale(loss/GRAD_ACCUM).backward()
                else: (loss/GRAD_ACCUM).backward()
                pending+=1; loss_sum+=loss.item()*y.numel(); token_sum+=y.numel()
                if pending==GRAD_ACCUM:
                    if SCALER: SCALER.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                    if SCALER: SCALER.step(opt); SCALER.update()
                    else: opt.step()
                    opt.zero_grad(set_to_none=True); sched.step(); pending=0; updates+=1
        if pending:
            if SCALER: SCALER.unscale_(opt)
            factor=GRAD_ACCUM/pending
            for p in model.parameters():
                if p.grad is not None: p.grad.mul_(factor)
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            if SCALER: SCALER.step(opt); SCALER.update()
            else: opt.step()
            opt.zero_grad(set_to_none=True); sched.step(); pending=0; updates+=1
        val_loss,val_n=evaluate(model,valid_docs)
        row={"epoch":epoch,"train_loss":loss_sum/max(token_sum,1),"validation_loss":val_loss,
             "validation_perplexity":math.exp(min(val_loss,20)),"validation_tokens":val_n,"updates":updates}
        history.append(row); save_ckpt(LATEST_PATH,model,opt,epoch,min(best,val_loss),history)
        if val_loss<best:
            best=val_loss; save_ckpt(BEST_PATH,model,opt,epoch,best,history)
        print(json.dumps(row,indent=2)); gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return history

if DEVICE.type=="cuda":
    training_history=train(MODEL,train_documents,valid_documents)
else:
    training_history=None
    print("Training is ready; enable a Kaggle GPU and rerun this cell.")

In [ ]:
def sample_token(logits,temperature=GEN_TEMPERATURE,top_k=GEN_TOP_K,top_p=GEN_TOP_P):
    logits=logits.float().clone()
    logits[:4]=-float("inf")
    if temperature<=0: return int(logits.argmax().item())
    logits/=temperature
    if top_k and top_k>0:
        kth=torch.topk(logits,min(top_k,logits.numel())).values[-1]
        logits[logits<kth]=-float("inf")
    if top_p and top_p<1:
        vals,idx=torch.sort(logits,descending=True)
        probs=torch.softmax(vals,dim=-1)
        remove=torch.cumsum(probs,0)-probs>top_p
        vals[remove]=-float("inf")
        filtered=torch.full_like(logits,-float("inf")); filtered.scatter_(0,idx,vals); logits=filtered
    return int(torch.multinomial(torch.softmax(logits,dim=-1),1).item())

@torch.inference_mode()
def generate_continuation(model,prompt,temperature=GEN_TEMPERATURE):
    model.eval()
    ids=tokenizer.encode(clean_text(prompt)) or [BOS_ID]
    memories=caches=None; position=0; logits=None
    for start in range(0,len(ids),SEGMENT_LENGTH):
        part=torch.tensor(ids[start:start+SEGMENT_LENGTH],dtype=torch.long,device=DEVICE)[None,:]
        with amp():
            logits,memories,caches=model(part,memories=memories,caches=caches,
                                         position_start=position,use_cache=True)
        position+=part.size(1)
    generated=[]; words=0
    for _ in range(MAX_GENERATION_TOKENS):
        nxt=sample_token(logits[0,-1],temperature)
        generated.append(nxt); words+=int(tokenizer.word_end[nxt])
        if words>=OUTPUT_WORDS: break
        step=torch.tensor([[nxt]],dtype=torch.long,device=DEVICE)
        with amp():
            logits,memories,caches=model(step,memories=memories,caches=caches,
                                         position_start=position,use_cache=True)
        position+=1
    text=tokenizer.decode(generated).strip()
    parts=text.split()
    if len(parts)>OUTPUT_WORDS: text=" ".join(parts[:OUTPUT_WORDS])
    if len(text.split())!=OUTPUT_WORDS:
        raise RuntimeError(f"Generation returned {len(text.split())} words, expected {OUTPUT_WORDS}.")
    return text

if BEST_PATH.exists():
    payload=torch.load(BEST_PATH,map_location=DEVICE,weights_only=True)
    if payload["tokenizer_sha256"]!=TOKENIZER_SHA256: raise ValueError("Checkpoint tokenizer does not match.")
    MODEL.load_state_dict(payload["model_state_dict"])
    print("Loaded best checkpoint:",BEST_PATH)
else:
    print("Train the model to create the best checkpoint.")

In [ ]:
def validation_pairs(docs,limit=8):
    result=[]
    for doc in docs:
        words=tokenizer.decode(doc).split()
        if len(words)>=512+OUTPUT_WORDS:
            result.append((" ".join(words[:512])," ".join(words[512:512+OUTPUT_WORDS])))
        if len(result)>=limit: break
    return result

if BEST_PATH.exists():
    preview=[]
    for prompt,reference in tqdm(validation_pairs(valid_documents,limit=32),desc="Validation preview"):
        pred=generate_continuation(MODEL,prompt)
        preview.append({"prompt":prompt,"reference":reference,"prediction":pred,
                        "prediction_words":len(pred.split())})
    (OUTPUT_DIR/"local_generation_preview.json").write_text(
        json.dumps(preview,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Saved",OUTPUT_DIR/"local_generation_preview.json")
else:
    print("No checkpoint yet; run training first.")

In [ ]:
if BEST_PATH.exists() and (OUTPUT_DIR/"local_generation_preview.json").exists():
    examples=json.loads((OUTPUT_DIR/"local_generation_preview.json").read_text(encoding="utf-8"))
    preds=[x["prediction"] for x in examples]
    refs=[x["reference"] for x in examples]
    def rouge_l_f1(pred,ref):
        a,b=pred.split(),ref.split()
        previous=[0]*(len(b)+1)
        for wa in a:
            current=[0]
            for j,wb in enumerate(b,1):
                current.append(previous[j-1]+1 if wa==wb else max(previous[j],current[-1]))
            previous=current
        common=previous[-1]
        precision=common/max(1,len(a)); recall=common/max(1,len(b))
        return 2*precision*recall/max(1e-12,precision+recall)
    def corpus_bleu4(candidates,references):
        from collections import Counter
        clipped=[0]*4; totals=[0]*4; clen=rlen=0
        for candidate,reference in zip(candidates,references):
            c,r=candidate.split(),reference.split(); clen+=len(c); rlen+=len(r)
            for n in range(1,5):
                cg=Counter(tuple(c[i:i+n]) for i in range(max(0,len(c)-n+1)))
                rg=Counter(tuple(r[i:i+n]) for i in range(max(0,len(r)-n+1)))
                totals[n-1]+=sum(cg.values())
                clipped[n-1]+=sum(min(v,rg[g]) for g,v in cg.items())
        precisions=[(clipped[i]+1)/(totals[i]+1) for i in range(4)]
        bp=1.0 if clen>rlen else math.exp(1-rlen/max(1,clen))
        return bp*math.exp(sum(math.log(v) for v in precisions)/4)
    metrics={
        "examples":len(examples),
        "ROUGE_L_F1":float(np.mean([rouge_l_f1(p,r) for p,r in zip(preds,refs)])),
        "BLEU_4":float(corpus_bleu4(preds,refs)),
    }
    try:
        from bert_score import score as bertscore
        _,_,f1=bertscore(preds,refs,lang="en",batch_size=2,verbose=False,
                         device=("cuda" if torch.cuda.is_available() else "cpu"))
        metrics["BERTScore_F1"]=float(f1.mean().item())
    except Exception as exc:
        metrics["BERTScore_error"]=str(exc)
        print("BERTScore could not run; ROUGE-L and BLEU-4 are still available.")
    (OUTPUT_DIR/"local_metrics.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")
    print(json.dumps(metrics,indent=2))
else:
    print("Run training and the held-out generation preview before local metric calculation.")

In [ ]:
SWEEP_TEMPERATURES = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
if BEST_PATH.exists():
    sweep_pairs = validation_pairs(valid_documents, limit=32)
    sweep_refs = [ref for _, ref in sweep_pairs]
    sweep_scores = {}
    from bert_score import score as _bertscore
    for temp in SWEEP_TEMPERATURES:
        outs = [generate_continuation(MODEL, prompt, temperature=temp)
                for prompt, _ in tqdm(sweep_pairs, desc=f"Sweep T={temp}")]
        _, _, f1 = _bertscore(outs, sweep_refs, lang="en", batch_size=2, verbose=False,
                              device=("cuda" if torch.cuda.is_available() else "cpu"))
        sweep_scores[temp] = float(f1.mean().item())
        print(f"T={temp}: BERTScore F1={sweep_scores[temp]:.4f}")
    GEN_TEMPERATURE = max(sweep_scores, key=sweep_scores.get)
    (OUTPUT_DIR/"temperature_sweep.json").write_text(json.dumps(
        {"scores": {str(k): v for k, v in sweep_scores.items()}, "selected": GEN_TEMPERATURE}, indent=2), encoding="utf-8")
    print("Selected GEN_TEMPERATURE:", GEN_TEMPERATURE)
else:
    print("Train the model first; no checkpoint for the temperature sweep.")

In [ ]:
from datasets import load_dataset

def extract_id(row,fallback):
    if isinstance(row,dict):
        for key in ("id","ID","example_id","document_id","doc_id"):
            if row.get(key) is not None: return str(row[key])
    return str(fallback)

def load_official_test():
    stream=load_dataset(DATASET_ID,DATASET_CONFIG,split="test",streaming=True,token=HF_TOKEN)
    rows=[]
    for idx,row in enumerate(tqdm(stream,desc="Official test prompts")):
        if isinstance(row,dict) and row and all(str(k).isdigit() for k in row):
            entries=((str(k),v) for k,v in row.items())
        else: entries=((extract_id(row,idx+1),row),)
        for ident,record in entries:
            prompt=clean_text(extract_text(record))
            if not prompt: raise ValueError(f"Empty prompt for ID {ident}")
            rows.append({"id":str(ident),"prompt":prompt})
        if len(rows)>=1000: break
    if not rows: raise RuntimeError("No official prompts loaded.")
    ids=[x["id"] for x in rows]
    if len(ids)!=len(set(ids)): raise ValueError("Duplicate official test IDs.")
    return rows

def save_submission(predictions,ids):
    if set(predictions)!=set(ids): raise ValueError("Prediction IDs do not match official test IDs.")
    for key,value in predictions.items():
        if len(value.split())!=OUTPUT_WORDS: raise ValueError(f"{key}: expected {OUTPUT_WORDS} words.")
    pred_path=OUTPUT_DIR/"prediction.json"
    pred_path.write_text(json.dumps({k:{"text":predictions[k]} for k in ids},ensure_ascii=False,indent=2),encoding="utf-8")
    zip_path=OUTPUT_DIR/"submission_phase1_multires_memory.zip"
    with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z: z.write(pred_path,arcname="prediction.json")
    return pred_path,zip_path

if RUN_OFFICIAL_GENERATION:
    if not BEST_PATH.exists(): raise FileNotFoundError("Train the model before official generation.")
    test_examples=load_official_test()
    test_ids=[x["id"] for x in test_examples]
    progress_path=OUTPUT_DIR/"official_progress.json"
    predictions={}
    if progress_path.exists():
        try:
            prior=json.loads(progress_path.read_text(encoding="utf-8"))
            if prior.get("tokenizer_sha256")==TOKENIZER_SHA256:
                predictions={str(k):v["text"] for k,v in prior.get("predictions",{}).items()
                             if isinstance(v,dict) and isinstance(v.get("text"),str)
                             and len(v["text"].split())==OUTPUT_WORDS}
        except Exception as e: print("Starting without previous partial output:",e)
    for item in tqdm(test_examples,desc="Official generation"):
        key=item["id"]
        if key in predictions: continue
        predictions[key]=generate_continuation(MODEL,item["prompt"],temperature=GEN_TEMPERATURE)
        if len(predictions)%10==0:
            progress_path.write_text(json.dumps({"tokenizer_sha256":TOKENIZER_SHA256,
                "predictions":{k:{"text":v} for k,v in predictions.items()}},ensure_ascii=False),encoding="utf-8")
    pred_path,zip_path=save_submission(predictions,test_ids)
    print("prediction.json:",pred_path)
    print("submission ZIP:",zip_path)
else:
    print("Set RUN_OFFICIAL_GENERATION=True in the configuration cell to run all official prompts.")